# TH3 - Spark RDD Movie Ratings

**Dữ liệu:**

1. `movies(2).txt` - **Schema**: `MovieID, Title, Genres`
2. `ratings_1(1).txt`, `ratings_2(1).txt` - **Schema**: `UserID, MovieID, Rating, Timestamp`
3. `users(1).txt` - **Schema**: `UserID, Gender, Age, Occupation, Zip-code`
4. `occupation.txt` - **Schema**: `ID, Occupation`

**Lưu ý:** Toàn bộ các bài bên dưới xử lý bằng Spark RDD.


## Khởi tạo SparkContext và đọc dữ liệu


In [1]:
from pathlib import Path
from datetime import datetime
import os
import sys
from pyspark import SparkConf, SparkContext

BASE_DIR = Path(r"D:\Project\hadoop\TH3")
MOVIES_PATH = BASE_DIR / "movies(2).txt"
RATINGS_1_PATH = BASE_DIR / "ratings_1(1).txt"
RATINGS_2_PATH = BASE_DIR / "ratings_2(1).txt"
USERS_PATH = BASE_DIR / "users(1).txt"
OCCUPATIONS_PATH = BASE_DIR / "occupation.txt"

required_files = [MOVIES_PATH, RATINGS_1_PATH, RATINGS_2_PATH, USERS_PATH, OCCUPATIONS_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Không tìm thấy file dữ liệu: " + ", ".join(missing_files))

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

if "sc" in globals():
    sc.stop()

conf = (
    SparkConf()
    .setMaster("local[*]")
    .setAppName("TH3 Spark RDD Movie Ratings")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
)
sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")

print("Spark version:", sc.version)
print("Base directory:", BASE_DIR)


Spark version: 4.1.1
Base directory: D:\Project\hadoop\TH3


## Parser và RDD dùng chung


In [2]:
def spark_file(path):
    return str(Path(path).resolve()).replace("\\", "/")


def parse_movie(line):
    movie_id_text, rest = line.strip().split(",", 1)
    title, genres_text = rest.rsplit(",", 1)
    genres = [genre.strip() for genre in genres_text.split("|") if genre.strip()]
    return int(movie_id_text), (title.strip(), genres)


def parse_rating(line):
    user_id, movie_id, rating, timestamp = line.strip().split(",")
    return int(user_id), int(movie_id), float(rating), int(timestamp)


def parse_user(line):
    user_id, gender, age, occupation_id, zip_code = line.strip().split(",")
    return int(user_id), gender, int(age), int(occupation_id), zip_code


def parse_occupation(line):
    occupation_id, occupation = line.strip().split(",", 1)
    return int(occupation_id), occupation.strip()


def sum_rating_counts(a, b):
    return a[0] + b[0], a[1] + b[1]


def average_with_count(value):
    total_rating, total_count = value
    return round(total_rating / total_count, 2), total_count


def print_table(title, rows, headers=None, limit=None):
    print("\n" + title)
    print("=" * len(title))
    if headers:
        print(" | ".join(headers))
        print("-" * max(20, len(" | ".join(headers))))
    selected_rows = rows[:limit] if limit else rows
    if not selected_rows:
        print("Không có dữ liệu.")
        return
    for row in selected_rows:
        print(" | ".join(str(value) for value in row))


movies = sc.textFile(spark_file(MOVIES_PATH)).map(parse_movie).cache()
ratings_1 = sc.textFile(spark_file(RATINGS_1_PATH)).map(parse_rating).cache()
ratings_2 = sc.textFile(spark_file(RATINGS_2_PATH)).map(parse_rating).cache()
ratings = ratings_1.union(ratings_2).cache()
users = sc.textFile(spark_file(USERS_PATH)).map(parse_user).cache()
occupations = sc.textFile(spark_file(OCCUPATIONS_PATH)).map(parse_occupation).cache()

movie_titles = movies.mapValues(lambda value: value[0]).cache()
movie_genres = movies.mapValues(lambda value: value[1]).cache()
ratings_by_movie = ratings.map(lambda row: (row[1], (row[2], 1))).cache()
ratings_by_user = ratings.map(lambda row: (row[0], (row[1], row[2], row[3]))).cache()
users_by_id = users.map(lambda row: (row[0], row[1:])).cache()

movies_count = movies.count()
ratings_1_count = ratings_1.count()
ratings_2_count = ratings_2.count()
ratings_count = ratings.count()
users_count = users.count()
occupation_count = occupations.count()

rating_movie_join_count = ratings.map(lambda row: (row[1], row)).join(movie_titles).count()
rating_user_join_count = ratings_by_user.join(users_by_id).count()

print("Số phim:", movies_count)
print("Số ratings file 1:", ratings_1_count)
print("Số ratings file 2:", ratings_2_count)
print("Tổng số ratings:", ratings_count)
print("Số users:", users_count)
print("Số occupations:", occupation_count)
print("Ratings join được với movies:", rating_movie_join_count)
print("Ratings join được với users:", rating_user_join_count)

assert ratings_count == ratings_1_count + ratings_2_count == 184
assert rating_movie_join_count == ratings_count
assert rating_user_join_count == ratings_count


Số phim: 50
Số ratings file 1: 84
Số ratings file 2: 100
Tổng số ratings: 184
Số users: 50
Số occupations: 15
Ratings join được với movies: 184
Ratings join được với users: 184


## Bài 1 - Điểm trung bình theo phim

Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim. Phần đề bài có chỗ ghi ngưỡng 50 lượt, còn phần giải pháp ghi 5 lượt; dữ liệu hiện tại chỉ có tối đa 18 lượt/phim, nên bên dưới kiểm tra ngưỡng 50 trước rồi in thêm kết quả với ngưỡng 5 để có kết quả minh họa.


In [3]:
movie_stats = (
    ratings_by_movie
    .reduceByKey(sum_rating_counts)
    .mapValues(average_with_count)
    .cache()
)

movie_stats_with_title = (
    movie_stats
    .join(movie_titles)
    .map(lambda item: (item[0], item[1][1], item[1][0][0], item[1][0][1]))
    .cache()
)

top_movies = (
    movie_stats_with_title
    .sortBy(lambda row: (-row[2], -row[3], row[1]))
    .take(15)
)

print_table(
    "Bài 1 - Điểm trung bình theo phim",
    top_movies,
    headers=["MovieID", "Title", "AvgRating", "TotalRatings"],
)

best_min_50 = (
    movie_stats_with_title
    .filter(lambda row: row[3] >= 50)
    .sortBy(lambda row: (-row[2], row[1]))
    .take(1)
)

print_table(
    "Phim điểm trung bình cao nhất với tối thiểu 50 lượt đánh giá",
    best_min_50,
    headers=["MovieID", "Title", "AvgRating", "TotalRatings"],
)

if not best_min_50:
    print("Ghi chú: Không có phim nào đạt tối thiểu 50 lượt đánh giá trong bộ dữ liệu này.")

best_min_5 = (
    movie_stats_with_title
    .filter(lambda row: row[3] >= 5)
    .sortBy(lambda row: (-row[2], row[1]))
    .take(1)
)

print_table(
    "Kết quả minh họa với tối thiểu 5 lượt đánh giá",
    best_min_5,
    headers=["MovieID", "Title", "AvgRating", "TotalRatings"],
)



Bài 1 - Điểm trung bình theo phim
MovieID | Title | AvgRating | TotalRatings
------------------------------------------
1015 | Sunset Boulevard (1950) | 4.36 | 7
1025 | The Terminator (1984) | 4.06 | 18
1013 | The Godfather: Part II (1974) | 4.0 | 17
1012 | Psycho (1960) | 4.0 | 2
1043 | No Country for Old Men (2007) | 3.89 | 18
1037 | The Lord of the Rings: The Fellowship of the Ring (2001) | 3.89 | 18
1047 | The Social Network (2010) | 3.86 | 7
1039 | The Lord of the Rings: The Return of the King (2003) | 3.82 | 11
1020 | E.T. the Extra-Terrestrial (1982) | 3.67 | 18
1040 | Gladiator (2000) | 3.61 | 18
1028 | Fight Club (1999) | 3.5 | 7
1050 | Mad Max: Fury Road (2015) | 3.47 | 18
1010 | Lawrence of Arabia (1962) | 3.44 | 18
1030 | The Silence of the Lambs (1991) | 3.14 | 7



Phim điểm trung bình cao nhất với tối thiểu 50 lượt đánh giá
MovieID | Title | AvgRating | TotalRatings
------------------------------------------
Không có dữ liệu.
Ghi chú: Không có phim nào đạt tối thiểu 50 lượt đánh giá trong bộ dữ liệu này.



Kết quả minh họa với tối thiểu 5 lượt đánh giá
MovieID | Title | AvgRating | TotalRatings
------------------------------------------
1015 | Sunset Boulevard (1950) | 4.36 | 7


## Bài 2 - Điểm trung bình theo thể loại


In [4]:
genre_rating_pairs = (
    ratings.map(lambda row: (row[1], row[2]))
    .join(movie_genres)
    .flatMap(lambda item: [(genre, (item[1][0], 1)) for genre in item[1][1]])
)

genre_stats = (
    genre_rating_pairs
    .reduceByKey(sum_rating_counts)
    .mapValues(average_with_count)
    .map(lambda item: (item[0], item[1][0], item[1][1]))
    .sortBy(lambda row: (-row[1], row[0]))
    .collect()
)

print_table(
    "Bài 2 - Điểm trung bình theo thể loại",
    genre_stats,
    headers=["Genre", "AvgRating", "TotalRatings"],
)



Bài 2 - Điểm trung bình theo thể loại
Genre | AvgRating | TotalRatings
--------------------------------
Film-Noir | 4.36 | 7
Horror | 4.0 | 2
Mystery | 4.0 | 2
Fantasy | 3.86 | 29
Crime | 3.81 | 42
Drama | 3.76 | 128
Sci-Fi | 3.73 | 54
Action | 3.71 | 54
Thriller | 3.7 | 27
Family | 3.67 | 18
Adventure | 3.63 | 83
Biography | 3.56 | 25


## Bài 3 - Điểm trung bình theo giới tính


In [5]:
user_gender = users.map(lambda row: (row[0], row[1])).cache()

movie_gender_stats = (
    ratings_by_user
    .join(user_gender)
    .map(lambda item: ((item[1][0][0], item[1][1]), (item[1][0][1], 1)))
    .reduceByKey(sum_rating_counts)
    .mapValues(average_with_count)
    .map(lambda item: (item[0][0], (item[0][1], item[1][0], item[1][1])))
    .join(movie_titles)
    .map(lambda item: (item[0], item[1][1], item[1][0][0], item[1][0][1], item[1][0][2]))
    .sortBy(lambda row: (row[1], row[2]))
    .take(30)
)

print_table(
    "Bài 3 - Điểm trung bình theo giới tính",
    movie_gender_stats,
    headers=["MovieID", "Title", "Gender", "AvgRating", "TotalRatings"],
)



Bài 3 - Điểm trung bình theo giới tính
MovieID | Title | Gender | AvgRating | TotalRatings
---------------------------------------------------
1020 | E.T. the Extra-Terrestrial (1982) | F | 3.55 | 10
1020 | E.T. the Extra-Terrestrial (1982) | M | 3.81 | 8
1028 | Fight Club (1999) | F | 3.5 | 3
1028 | Fight Club (1999) | M | 3.5 | 4
1040 | Gladiator (2000) | F | 3.64 | 7
1040 | Gladiator (2000) | M | 3.59 | 11
1010 | Lawrence of Arabia (1962) | F | 3.31 | 8
1010 | Lawrence of Arabia (1962) | M | 3.55 | 10
1050 | Mad Max: Fury Road (2015) | F | 3.32 | 14
1050 | Mad Max: Fury Road (2015) | M | 4.0 | 4
1043 | No Country for Old Men (2007) | F | 3.83 | 6
1043 | No Country for Old Men (2007) | M | 3.92 | 12
1012 | Psycho (1960) | F | 4.0 | 2
1015 | Sunset Boulevard (1950) | F | 4.5 | 1
1015 | Sunset Boulevard (1950) | M | 4.33 | 6
1013 | The Godfather: Part II (1974) | F | 3.94 | 8
1013 | The Godfather: Part II (1974) | M | 4.06 | 9
1037 | The Lord of the Rings: The Fellowship of the Ring (

## Bài 4 - Điểm trung bình theo nhóm tuổi


In [6]:
def age_group(age):
    if age < 18:
        return "<18"
    if age <= 24:
        return "18-24"
    if age <= 34:
        return "25-34"
    if age <= 44:
        return "35-44"
    if age <= 49:
        return "45-49"
    if age <= 55:
        return "50-55"
    return "56+"


user_age_group = users.map(lambda row: (row[0], age_group(row[2]))).cache()

movie_age_group_stats = (
    ratings_by_user
    .join(user_age_group)
    .map(lambda item: ((item[1][0][0], item[1][1]), (item[1][0][1], 1)))
    .reduceByKey(sum_rating_counts)
    .mapValues(average_with_count)
    .map(lambda item: (item[0][0], (item[0][1], item[1][0], item[1][1])))
    .join(movie_titles)
    .map(lambda item: (item[0], item[1][1], item[1][0][0], item[1][0][1], item[1][0][2]))
    .sortBy(lambda row: (row[1], row[2]))
    .take(30)
)

print_table(
    "Bài 4 - Điểm trung bình theo nhóm tuổi",
    movie_age_group_stats,
    headers=["MovieID", "Title", "AgeGroup", "AvgRating", "TotalRatings"],
)



Bài 4 - Điểm trung bình theo nhóm tuổi
MovieID | Title | AgeGroup | AvgRating | TotalRatings
-----------------------------------------------------
1020 | E.T. the Extra-Terrestrial (1982) | 18-24 | 3.5 | 2
1020 | E.T. the Extra-Terrestrial (1982) | 25-34 | 3.58 | 6
1020 | E.T. the Extra-Terrestrial (1982) | 35-44 | 3.81 | 8
1020 | E.T. the Extra-Terrestrial (1982) | 45-49 | 4.0 | 1
1020 | E.T. the Extra-Terrestrial (1982) | 56+ | 3.0 | 1
1028 | Fight Club (1999) | 25-34 | 3.5 | 3
1028 | Fight Club (1999) | 35-44 | 3.5 | 2
1028 | Fight Club (1999) | 50-55 | 3.5 | 2
1040 | Gladiator (2000) | 18-24 | 3.5 | 1
1040 | Gladiator (2000) | 25-34 | 3.42 | 6
1040 | Gladiator (2000) | 35-44 | 3.75 | 6
1040 | Gladiator (2000) | 45-49 | 4.0 | 2
1040 | Gladiator (2000) | 50-55 | 3.75 | 2
1040 | Gladiator (2000) | 56+ | 3.0 | 1
1010 | Lawrence of Arabia (1962) | 25-34 | 3.6 | 5
1010 | Lawrence of Arabia (1962) | 35-44 | 3.28 | 9
1010 | Lawrence of Arabia (1962) | 45-49 | 3.5 | 2
1010 | Lawrence of Ar

## Bài 5 - Điểm trung bình theo nghề nghiệp


In [7]:
user_occupation_id = users.map(lambda row: (row[0], row[3])).cache()
occupation_names = occupations.cache()

occupation_stats = (
    ratings_by_user
    .join(user_occupation_id)
    .map(lambda item: (item[1][1], (item[1][0][1], 1)))
    .reduceByKey(sum_rating_counts)
    .mapValues(average_with_count)
    .join(occupation_names)
    .map(lambda item: (item[0], item[1][1], item[1][0][0], item[1][0][1]))
    .sortBy(lambda row: (-row[2], row[1]))
    .collect()
)

print_table(
    "Bài 5 - Điểm trung bình theo nghề nghiệp",
    occupation_stats,
    headers=["OccupationID", "Occupation", "AvgRating", "TotalRatings"],
)



Bài 5 - Điểm trung bình theo nghề nghiệp
OccupationID | Occupation | AvgRating | TotalRatings
----------------------------------------------------
1 | Programmer | 4.25 | 10
12 | Designer | 4.0 | 13
15 | Student | 4.0 | 8
14 | Consultant | 3.86 | 14
8 | Nurse | 3.86 | 11
11 | Journalist | 3.85 | 17
6 | Artist | 3.73 | 11
4 | Teacher | 3.7 | 5
2 | Doctor | 3.69 | 21
5 | Lawyer | 3.65 | 17
9 | Salesperson | 3.65 | 17
10 | Accountant | 3.58 | 6
3 | Engineer | 3.56 | 18
7 | Manager | 3.47 | 16


## Bài 6 - Điểm trung bình theo năm


In [8]:
def year_from_unix(timestamp):
    return datetime.utcfromtimestamp(timestamp).year


year_stats = (
    ratings
    .map(lambda row: (year_from_unix(row[3]), (row[2], 1)))
    .reduceByKey(sum_rating_counts)
    .mapValues(average_with_count)
    .map(lambda item: (item[0], item[1][0], item[1][1]))
    .sortBy(lambda row: row[0])
    .collect()
)

print_table(
    "Bài 6 - Điểm trung bình theo năm",
    year_stats,
    headers=["Year", "AvgRating", "TotalRatings"],
)

print("\nRun all success: Hoàn thành 6 bài Spark RDD.")



Bài 6 - Điểm trung bình theo năm
Year | AvgRating | TotalRatings
-------------------------------
2020 | 3.75 | 184

Run all success: Hoàn thành 6 bài Spark RDD.
